# V6.5-19 — Data v5 (v4 + insulator bổ sung) + teacher fine-tune → distill YOLO26n (Raspberry Pi)
Mục tiêu: **mAP50 MEAN_5 > 0.82** (5 class lỗi) trên `val_clean`, model chạy trên Raspberry Pi 3 (CPU, NCNN).

**Dataset `datasets/v6p5_v5`** = v4 + rule I: box insulator bổ sung cho ảnh chưa có nhãn insulator
(Grounding DINO + OWLv2 phải đồng ý, ngưỡng chọn trên benchmark 1.000 ảnh có GT: `data/label_audit/insulator_ov/`).
Chạy tự động bằng `pipeline_v5.sh` (train → eval → upload Camber → push code → tắt studio).

**Thay đổi so với v3** (chi tiết `data/label_audit/README.md`):
- **Dataset giữ đủ 7 class** (bộ chung cho YOLO / RT-DETR / StructDETR — StructDETR cần insulator làm ngữ cảnh).
  **YOLO học cả insulator** (class ngữ cảnh, giống V6.4) + 5 class lỗi + person. Lưu ý: nhãn insulator còn thiếu ở ảnh
  cận cảnh → v3 (có học insulator) thấp hơn v2 ~2 điểm ở class lỗi; điền nhãn insulator là việc tiếp theo.
  Muốn bỏ class nào khỏi lúc train: sửa `TRAIN_CLASSES` ở cell 1 (dataset không đổi).
- **Thêm class `person`** (YOLO11x COCO, conf ≥ 0.5, mọi split): người/thợ điện là nguồn FO báo nhầm lớn nhất.
  Trên Pi: `person` **có cảnh báo** (loại riêng "có người trên/gần đường dây", không phải FO).
- **FO không thắt bằng SAM nữa** (SAM cắt dây/đuôi → quy ước lẫn lộn); thêm 1 FO bị thiếu nhãn ở val_clean.

**Tiết kiệm GPU** (~5 giờ thay vì ~10 giờ):
- Teacher khởi tạo từ **teacher v2** (5 class lỗi, tốt nhất) + head thêm `person` → fine-tune 25 epoch.
- Student khởi tạo từ **student v3** (đổi thứ tự class trong head) → distill 30 epoch.
- Cache ảnh `.npy` dùng lại của v3 (symlink) → epoch đầu không phải tạo cache 53 GB.
- Cell cuối (tùy chọn): **tự tắt studio** khi xong để không tốn tiền GPU lúc ngủ.

**Thứ tự**: 1 → 2 → 3 (teacher) → 4 (student) → 5 (NCNN) → 6 (eval) → 8 (tự tắt, nếu bật). Bị ngắt thì chạy lại đúng cell đó (tự resume).


In [ ]:
# 1. CONFIG + GATE (không train)
from pathlib import Path
import os, csv, time, logging
import torch, yaml, ultralytics
from ultralytics import YOLO
from ultralytics.utils import LOGGER

candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT = next(p for p in candidates if (p / "pyproject.toml").is_file() and (p / "legacy/notebooks/v6p5").is_dir())
WEIGHTS = PROJECT / "legacy/notebooks/v6p5/weights"
DATASET = PROJECT.parent / "datasets/v6p5_v5"
DATA_YAML, VAL_CLEAN_YAML = DATASET / "data.yaml", DATASET / "data_val_clean.yaml"
OUT = PROJECT / "artifacts/v6p5_v5"
TEACHER_SRC = PROJECT / "artifacts/v6p5_v2/runs/teacher/weights/best.pt"  # 5 class lỗi (cùng thứ tự)
STUDENT_SRC = PROJECT / "artifacts/v6p5_v3/runs/student_distill/weights/best.pt"  # insulator + 5 class lỗi
RUNS = OUT / "runs"
NAMES = ["insulator", "insulator_broken", "insulator_flashover", "bird_nest", "foreign_object", "broken_strand", "person"]
DEFECTS = NAMES[1:6]  # MEAN_5 = thước đo chính (5 class lỗi); person có cảnh báo nhưng báo cáo AP riêng
TRAIN_CLASSES = None  # None = học cả 7 class (insulator là class ngữ cảnh); ví dụ [0, 1, 2, 3, 4, 5] = bỏ person
TARGET_MAP50 = 0.82
IMGSZ, DEVICE, SEED = 640, 0, 20261012
WORKERS = min(8, os.cpu_count() or 4)
os.environ.setdefault("PYTHONHASHSEED", str(SEED))

# Công thức chung cho teacher và student (AdamW + cosine; plateau ở v6p5-14 giảm LR quá muộn)
COMMON = dict(data=str(DATA_YAML), imgsz=IMGSZ, device=DEVICE, workers=WORKERS, seed=SEED, deterministic=True,
              optimizer="AdamW", lr0=1e-3, lrf=0.05, cos_lr=True, weight_decay=1e-3, warmup_epochs=3.0,
              close_mosaic=15, cache="disk", amp=True, pretrained=True, plots=False, save_period=5,
              classes=TRAIN_CLASSES)

if not torch.cuda.is_available():
    raise RuntimeError("Chưa có GPU — chuyển studio sang GPU rồi chạy lại.")
if ultralytics.__version__ != "8.4.172":
    raise RuntimeError(f"Cần ultralytics==8.4.172 (có distill_model); hiện tại={ultralytics.__version__}")
import ncnn  # export cho Raspberry Pi; thiếu thì: pip install ncnn pnnx

cfg = yaml.safe_load(DATA_YAML.read_text())
assert [cfg["names"][k] for k in sorted(cfg["names"])] == NAMES
for split in ["images/train", "images/val", "images/val_clean"]:
    print(split, sum(1 for p in (DATASET / split).iterdir() if p.suffix != ".npy"), "ảnh")
for w in ["yolo26m.pt", "yolo26n.pt"]:
    assert (WEIGHTS / w).is_file(), w
assert TEACHER_SRC.is_file() and STUDENT_SRC.is_file()
assert yaml.safe_load((DATASET / "manifest.json").read_text())["params"]["add_insulator"], "dataset chưa bổ sung insulator (v5)"
OUT.mkdir(parents=True, exist_ok=True)
print("GPU:", torch.cuda.get_device_name(0), "| Ultralytics:", ultralytics.__version__, "| workers:", WORKERS)

In [ ]:
# 2. HÀM TRAIN: resume / bỏ qua nếu xong / watchdog chống treo / tự thử lại, in 1 dòng mỗi epoch
import gc, shutil, signal, threading
def _epoch_line(trainer):
    m, loss = trainer.metrics, trainer.label_loss_items(trainer.tloss)
    # sau epoch cuối ultralytics val lại bằng best.pt và gọi lại callback này
    final = trainer.stop and getattr(trainer, "_stop_seen", False)
    trainer._stop_seen = trainer.stop
    tag = "best.pt (val cuối)" if final else f"ep {trainer.epoch + 1:>3}/{trainer.epochs}"
    print(f"{trainer.args.name} {tag}  "
          f"box {float(loss['train/box_loss']):.3f} cls {float(loss['train/cls_loss']):.3f}  "
          f"mAP50 {m.get('metrics/mAP50(B)', 0):.4f}  "
          f"mAP50-95 {m.get('metrics/mAP50-95(B)', 0):.4f}  lr {trainer.optimizer.param_groups[0]['lr']:.2e}  "
          f"{time.strftime('%H:%M')}", flush=True)

HANG_MINUTES = 15  # không có batch nào chạy xong trong khoảng này → coi là treo
MAX_RETRIES = 3

class Watchdog:
    """Gửi SIGINT cho kernel nếu train không còn tiến triển (giống bấm Interrupt) để cell tự resume."""
    def __init__(self, minutes):
        self.limit, self.beat, self.armed, self.fired = minutes * 60, time.time(), False, False
        self._stop = threading.Event()
    def heartbeat(self, *_):  # gọi sau mỗi batch train/val
        self.beat, self.armed = time.time(), True
    def _run(self):
        while not self._stop.wait(30):
            if self.armed and time.time() - self.beat > self.limit:
                self.fired = True
                print(f"[watchdog] {self.limit // 60} phút không có batch nào chạy xong → ngắt để resume", flush=True)
                os.kill(os.getpid(), signal.SIGINT)
                return
    def __enter__(self):
        threading.Thread(target=self._run, daemon=True).start()
        return self
    def __exit__(self, *exc):
        self._stop.set()

def _latest_checkpoint(run):
    """last.pt, nếu hỏng (bị kill lúc đang ghi) thì lùi về epochN.pt mới nhất."""
    epochs = sorted(run.glob("weights/epoch*.pt"), key=lambda p: -int(p.stem[5:]))
    for p in [run / "weights/last.pt", *epochs]:
        if p.is_file():
            try:
                return p, torch.load(p, map_location="cpu", weights_only=False)
            except Exception as e:
                print(f"[{run.name}] checkpoint hỏng, bỏ qua {p.name}: {e}")
    return None, None

def train_stage(name, model, **kw):
    run = RUNS / name
    best = run / "weights/best.pt"
    for attempt in range(1, MAX_RETRIES + 2):
        ckpt, ck = _latest_checkpoint(run)
        done = ck is not None and (ck.get("epoch", -1) == -1  # ultralytics đánh dấu run đã xong bằng epoch=-1
                                   or ck["epoch"] + 1 >= ck.get("train_args", {}).get("epochs", 10**9))
        if done and (ck.get("train_args", {}).get("epochs") == kw.get("epochs", ck.get("train_args", {}).get("epochs"))
                     or ck.get("epoch", -1) == -1):
            print(f"[{name}] đã xong, bỏ qua: {best}")
            return best
        if ck is not None and "epochs" in kw and ck.get("train_args", {}).get("epochs") != kw["epochs"]:
            print(f"[{name}] run cũ train {ck['train_args']['epochs']} epoch ≠ {kw['epochs']} → bắt đầu lại")
            ck = None
        if ck is not None:
            print(f"[{name}] resume từ {ckpt.name} (đã xong {ck['epoch'] + 1} epoch)")
            yolo, args = YOLO(str(ckpt)), dict(resume=True)
        else:
            if run.is_dir():  # không có checkpoint dùng được → làm lại từ đầu
                shutil.rmtree(run)
            print(f"[{name}] bắt đầu: {model}")
            yolo, args = YOLO(str(model)), dict(project=str(RUNS), name=name, exist_ok=False, **{**COMMON, **kw})
        wd = Watchdog(HANG_MINUTES)
        for event in ("on_train_batch_end", "on_val_batch_end", "on_fit_epoch_end"):
            yolo.add_callback(event, wd.heartbeat)
        yolo.add_callback("on_fit_epoch_end", _epoch_line)
        level = LOGGER.level
        LOGGER.setLevel(logging.WARNING)  # tắt progress bar + log chi tiết → output không bị treo
        try:
            with wd:
                yolo.train(**args)
            print(f"[{name}] xong: {best}")
            return best
        except KeyboardInterrupt:
            if not wd.fired:
                print(f"[{name}] dừng theo yêu cầu. Chạy lại cell này để resume.")
                raise
            print(f"[{name}] treo → resume (lần {attempt}/{MAX_RETRIES})")
        except Exception as e:
            print(f"[{name}] lỗi {type(e).__name__}: {e} → resume (lần {attempt}/{MAX_RETRIES})")
        finally:
            LOGGER.setLevel(level)
            del yolo
            gc.collect()
            torch.cuda.empty_cache()
        time.sleep(30)
    raise RuntimeError(f"[{name}] thất bại sau {MAX_RETRIES} lần thử lại — xem log ở trên")

In [ ]:
# 2b. KHỞI TẠO TỪ MODEL CŨ: sắp xếp lại head theo class v4 (không train, vài giây, CPU)
import copy

def remap_head(src, dst, order, names):
    """Ghi checkpoint có head theo thứ tự class mới.
    order[i] = chỉ số class cũ cho class mới i; None = class mới (weight 0, bias thấp → ban đầu không đoán class này)."""
    if dst.is_file():
        return dst
    ck = torch.load(src, map_location="cpu", weights_only=False)
    m = copy.deepcopy(ck.get("ema") or ck["model"]).float()
    head = m.model[-1]
    for branch in [head.cv3, *([head.one2one_cv3] if hasattr(head, "one2one_cv3") else [])]:
        for seq in branch:
            old = seq[-1]  # Conv2d cuối của nhánh class: out_channels = nc
            new = torch.nn.Conv2d(old.in_channels, len(order), 1, bias=True)
            with torch.no_grad():
                for i, j in enumerate(order):
                    if j is None:
                        new.weight[i].zero_()
                        new.bias[i] = old.bias.min()
                    else:
                        new.weight[i], new.bias[i] = old.weight[j], old.bias[j]
            seq[-1] = new
    head.nc = len(order)
    head.no = head.nc + head.reg_max * 4
    m.yaml["nc"], m.nc, m.names = head.nc, head.nc, dict(enumerate(names))
    dst.parent.mkdir(parents=True, exist_ok=True)
    torch.save({"model": m.half(), "ema": None, "epoch": -1, "optimizer": None, "train_args": {},
                "date": ck.get("date"), "version": ck.get("version")}, dst)
    return dst

INIT = OUT / "init"
TEACHER_INIT = remap_head(TEACHER_SRC, INIT / "teacher_v2_plus_person.pt", [None, 0, 1, 2, 3, 4, None], NAMES)
STUDENT_INIT = remap_head(STUDENT_SRC, INIT / "student_v3_reordered.pt", [0, 1, 2, 3, 4, 5, None], NAMES)
for w in [TEACHER_INIT, STUDENT_INIT]:
    print(w.name, YOLO(str(w)).names)


In [ ]:
# 3. TEACHER — YOLO26m từ teacher v2 + head person, fine-tune 25 epoch (~2,5 giờ trên L4)
TEACHER_BEST = train_stage("teacher", TEACHER_INIT, epochs=25, batch=24, patience=10, close_mosaic=8, warmup_epochs=1.0)


In [ ]:
# 4. STUDENT — YOLO26n từ student v3 (head đổi thứ tự), distill từ teacher v4, 30 epoch (~2,7 giờ trên L4)
TEACHER_BEST = RUNS / "teacher/weights/best.pt"
assert TEACHER_BEST.is_file(), "Chạy cell 3 (teacher) trước"
STUDENT_BEST = train_stage("student_distill", STUDENT_INIT, epochs=30, batch=32, patience=10, close_mosaic=8,
                           warmup_epochs=1.0, distill_model=str(TEACHER_BEST))


In [ ]:
# 5. EXPORT NCNN cho Raspberry Pi
STUDENT_BEST = RUNS / "student_distill/weights/best.pt"
NCNN_DIR = STUDENT_BEST.with_name("best_ncnn_model")
if not NCNN_DIR.is_dir():
    YOLO(str(STUDENT_BEST)).export(format="ncnn", imgsz=IMGSZ, half=False)
print("Copy thư mục này sang Pi:", NCNN_DIR)
print("Trên Pi: pip install ultralytics ncnn; YOLO('best_ncnn_model', task='detect').predict('anh.jpg', imgsz=640)")

In [ ]:
# 6. EVAL trên val gốc + val_clean, kiểm tra mục tiêu
import pandas as pd

MODELS = {
    "teacher_v2_init": OUT / "init/teacher_v2_plus_person.pt",  # điểm xuất phát, để thấy fine-tune được bao nhiêu
    "teacher_y26m": RUNS / "teacher/weights/best.pt",
    "student_distill": RUNS / "student_distill/weights/best.pt",
    "student_distill_ncnn": RUNS / "student_distill/weights/best_ncnn_model",  # bản chạy trên Pi (CPU)
}
rows = []
for vname, vyaml in [("val", DATA_YAML), ("val_clean", VAL_CLEAN_YAML)]:
    for tag, w in MODELS.items():
        if not w.exists():
            continue
        ncnn_model = w.is_dir()
        b = YOLO(str(w), task="detect").val(data=str(vyaml), imgsz=IMGSZ, batch=1 if ncnn_model else 16,
                                            device="cpu" if ncnn_model else DEVICE, plots=False, verbose=False, classes=TRAIN_CLASSES,
                                            project=str(OUT / "eval"), name=f"{tag}_{vname}", exist_ok=True).box
        rows.append({"model": tag, "val": vname, "class": "ALL", "P": float(b.mp), "R": float(b.mr), "AP50": float(b.map50), "AP50_95": float(b.map)})
        for i, (c, a50, a) in enumerate(zip(b.ap_class_index, b.ap50, b.ap)):
            rows.append({"model": tag, "val": vname, "class": NAMES[int(c)], "P": float(b.p[i]), "R": float(b.r[i]), "AP50": float(a50), "AP50_95": float(a)})
res = pd.DataFrame(rows)
mean5 = res[res["class"].isin(DEFECTS)].groupby(["model", "val"], as_index=False)[["P", "R", "AP50", "AP50_95"]].mean()
res = pd.concat([res, mean5.assign(**{"class": "MEAN_5"})], ignore_index=True)
res.to_csv(OUT / "eval.csv", index=False)
display(res.pivot_table(index=["val", "class"], columns="model", values=["P", "R", "AP50", "AP50_95"]).round(4))
pi = res.query("model == 'student_distill_ncnn' and val == 'val_clean' and `class` == 'MEAN_5'")
if len(pi):
    v = float(pi.AP50.iloc[0])
    print(f"Model Raspberry Pi (NCNN) trên val_clean: mAP50 MEAN_5 = {v:.4f} →",
          "ĐẠT" if v > TARGET_MAP50 else "CHƯA ĐẠT", f"mục tiêu {TARGET_MAP50}")

In [ ]:
# 7. XEM TIẾN ĐỘ từ results.csv (khi mở lại notebook / kernel đã restart; không ảnh hưởng train)
import datetime as dt
for stage in ["teacher", "student_distill"]:
    f = RUNS / stage / "results.csv"
    if not f.exists():
        continue
    rows_ = [{k.strip(): v for k, v in r.items()} for r in csv.DictReader(f.open())]
    best = max(rows_, key=lambda r: float(r["metrics/mAP50(B)"]))
    print(f"== {stage}: {len(rows_)} epoch | best mAP50 {float(best['metrics/mAP50(B)']):.4f} (ep {best['epoch']}) "
          f"| cập nhật {dt.datetime.fromtimestamp(f.stat().st_mtime):%H:%M:%S}")
    for r in rows_[-10:]:
        print(f"  ep {r['epoch']:>3}  box {float(r['train/box_loss']):.3f}  mAP50 {float(r['metrics/mAP50(B)']):.4f}  "
              f"mAP50-95 {float(r['metrics/mAP50-95(B)']):.4f}  lr {float(r['lr/pg0']):.2e}")

In [ ]:
# 8. (TÙY CHỌN) TỰ TẮT STUDIO khi chạy xong để không tốn tiền GPU — đặt AUTO_STOP = True rồi chạy
# Mẹo: chạy cell 3 → 4 → 5 → 6 → 8 liên tiếp bằng "Run Below" / chọn nhiều cell trước khi đi ngủ.
AUTO_STOP = False  # pipeline_v5.sh tự tắt studio SAU KHI upload Camber + push code
if AUTO_STOP:
    from lightning_sdk import Studio
    print("Tắt studio lúc", time.strftime("%H:%M"), "— kết quả ở", OUT)
    Studio().stop()
